# Silver — CRM Product Info
Product master data from the CRM.

`bronze.crm_prd_info` → `silver.crm_products`

## Init

In [ ]:
import os, sys
import pyspark.sql.functions as F
from pyspark.sql.functions import col
from pyspark.sql.types import DateType
from pyspark.sql.window import Window

# Make src/ importable from wherever this notebook runs (git folder, bundle, VS Code sync)
root = os.getcwd()
while not os.path.isdir(os.path.join(root, "src")) and root != "/":
    root = os.path.dirname(root)
sys.path.insert(0, os.path.join(root, "src"))

from lakehouse.transforms import trim_strings, normalize, rename, yyyymmdd_to_date

CATALOG = "workspace"

## Read bronze table

In [ ]:
df = spark.table(f"{CATALOG}.bronze.crm_prd_info")

## Transformations

### Trim all string columns

In [ ]:
df = trim_strings(df)

### Parse the product key
`prd_key` looks like `CO-RF-FR-R92B-58`. The first 5 chars (`CO-RF` → `CO_RF`) are the category id used by the ERP; the rest is the real product number.

In [ ]:
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))

### Default missing cost to 0

In [ ]:
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))

### Normalize product line

In [ ]:
df = normalize(df, "prd_line", {"M": "Mountain", "R": "Road", "S": "Other Sales", "T": "Touring"})

### Cast dates

In [ ]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

### Rename to business-friendly names

In [ ]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
df = rename(df, RENAME_MAP)

## Sanity check

In [ ]:
df.limit(10).display()

## Write silver table

In [ ]:
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"{CATALOG}.silver.crm_products")

In [ ]:
%sql
SELECT * FROM workspace.silver.crm_products LIMIT 10;